# Initialization setup

In [1]:
# del pip_install
try:
  if pip_install:
    pass
  else:
    raise NameError
except NameError as err:
  !pip uninstall -y transformers huggingface-hub datasets
  !pip install "huggingface-hub>=0.34.0,<1.0"
  !pip install "transformers>=4.40,<4.46"
  !pip install datasets evaluate fsspec
  pip_install=True

Found existing installation: transformers 4.45.2
Uninstalling transformers-4.45.2:
  Successfully uninstalled transformers-4.45.2
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
evaluate 0.4.6 requires datasets>=2.0.0, which is not installed.
torchtune 0.6.1 requires datasets, which is not installed.
peft 0.18.1 requires transformers, which is not installed.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, which is not installed.
  Using cached transformers-4.45.2

In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

from transformers import (
    BertConfig,
    BertTokenizer,
    BertModel,
    BertForMaskedLM,
    BertPreTrainedModel,
    BertForQuestionAnswering,
)
from transformers.activations import ACT2FN
from transformers.modeling_outputs import (
    CausalLMOutputWithCrossAttentions,
    BaseModelOutputWithCrossAttentions,
    BaseModelOutputWithPoolingAndCrossAttentions,
)
from transformers.pytorch_utils import apply_chunking_to_forward

try:
    from transformers.pytorch_utils import (
        find_pruneable_heads_and_indices,
        prune_linear_layer,
    )
except ImportError:
    from transformers.modeling_utils import (
        find_pruneable_heads_and_indices,
        prune_linear_layer,
    )

from transformers.modeling_outputs import QuestionAnsweringModelOutput

from datasets import load_dataset
from transformers import BertTokenizer, BertForMaskedLM
from torch.nn.functional import softmax

from tqdm import tqdm


from datasets import load_dataset
from transformers import BertTokenizerFast
import torch
from tqdm import tqdm
import evaluate

import os
os.environ["HF_DATASETS_OFFLINE"] = "0"

# Quantization Components

## Observers

In [3]:
class ObserverBase(nn.Module):
    def __init__(self, dtype=torch.qint8, qscheme=torch.per_tensor_affine):
        """
        ObserverBase class is a base class for observers in PyTorch quantization.

        Args:
            dtype (torch.dtype): Data type for quantization, typically `torch.qint8` or `torch.quint8`.
            qscheme (torch.qscheme): Quantization scheme. For example, `torch.per_tensor_affine`.
        """
        super(ObserverBase, self).__init__()
        self.dtype = dtype
        self.qscheme = qscheme
        self.is_symmetric = qscheme in [torch.per_tensor_symmetric, torch.per_channel_symmetric]

    def forward(self, x):
        """
        Placeholder for the forward method that will process the input data (activations or weights).
        To be overridden in derived classes.

        Args:
            x (torch.Tensor): The input tensor to observe.
        """
        raise NotImplementedError("ObserverBase.forward must be implemented in derived classes")

    def calculate_qparams(self):
        """
        Placeholder for the calculate_qparams method. It should calculate the quantization parameters
        (scale and zero point) based on the statistics collected in the forward pass.
        """
        raise NotImplementedError("ObserverBase.calculate_qparams must be implemented in derived classes")

In [4]:

class MinMaxObserver(ObserverBase):
    def __init__(self,
                 dtype=torch.qint8,
                 qscheme=torch.per_tensor_affine,
                 nof_bits=8,
                 is_calibrate=False,
                 name='base'):
        """
        MinMaxObserver class is used to record the minimum and maximum values of the input activations.
        These values will be used to compute the scale and zero-point for quantization.

        Args:
            dtype (torch.dtype): The data type for quantization, e.g., torch.qint8.
            qscheme (torch.qscheme): The quantization scheme, e.g., torch.per_tensor_affine.
            nof_bits (int): Number of bits for quantization.
            is_calibrate (bool): Flag indicating whether to calculate the scale and zero-point during forwarding.
        """
        super(MinMaxObserver, self).__init__()
        self.dtype = dtype
        self.qscheme = qscheme
        self.nof_bits = nof_bits
        self.is_calibrate = is_calibrate
        self.min_val = torch.tensor(float('inf'))  # Initialize min with infinity
        self.max_val = torch.tensor(float('-inf'))  # Initialize max with -infinity
        self.zero_point = None
        self.scale = None
        self.name = name
        self.ez = None
        self.nml = None
        self.sz  = None

    def forward(self, x):
        """
        This method updates the minimum and maximum values by comparing the current min and max
        with those in the input tensor. If is_calibrate is True, it calculates the scale and zero-point.

        Args:
            x (torch.Tensor): The input tensor to observe.
        """
        global_min = x.min()
        global_max = x.max()

        if global_min < self.min_val:
            self.min_val = global_min
        if global_max > self.max_val:
            self.max_val = global_max

        # Calculate scale and zero-point during forwarding if is_calibrate is True
        self.calculate_qparams()
        self.ez = x.element_size()
        self.nml = x.numel()
        self.sz = x.size()

        return x

    def calculate_qparams(self):
        """
        This method computes the scale and zero-point for quantization based on the min and max values
        observed during the forward pass.

        Returns:
            scale (torch.Tensor): The quantization scale.
            zero_point (torch.Tensor): The quantization zero-point.
        """
        if self.min_val == self.max_val:
            # To handle cases where min == max to avoid division by zero
            self.scale = torch.tensor([1.0], dtype=torch.float32)
            self.zero_point = torch.tensor([0], dtype=torch.int32)
        else:
            # Calculate scale and zero-point for qint8 quantization
            self.scale = (self.max_val - self.min_val) / (2 ** self.nof_bits - 1)
            if self.qscheme == torch.per_tensor_affine:
                self.zero_point = torch.round(-self.min_val / self.scale).clamp(0, 2 ** self.nof_bits - 1)
            else:
                self.zero_point = 0
        return self.scale, self.zero_point

    def quantizer(self, x, scale=None):
        """
        Quantizes the input tensor using the calculated scale and zero-point.

        Args:
            x (torch.Tensor): The input tensor to quantize.

        Returns:
            torch.Tensor: The quantized tensor.
        """
        # if self.is_calibrate:
        if self.scale is None or self.zero_point is None:
            raise ValueError(f"Scale and zero-point must be calculated before quantizing [{self.name}].")

        # Quantize the input
        if self.qscheme == torch.per_tensor_affine:
            x_q = (x / (self.scale) + self.zero_point).round().clamp(0, 2 ** self.nof_bits - 1)
        else:
            x_q = (x / (self.scale) + self.zero_point).round().clamp(-2 ** (self.nof_bits-1), 2 ** (self.nof_bits-1) - 1)

        return x_q.int()

    def dequantizer(self, x_q):
        """
        Dequantizes the input tensor using the calculated scale and zero-point.

        Args:
            x_q (torch.Tensor): The quantized tensor to dequantize.

        Returns:
            torch.Tensor: The dequantized tensor.
        """

        if self.scale is None or self.zero_point is None:
            raise ValueError(f"Scale and zero-point must be calculated before dequantizing [{self.name}].")

        # Dequantize the input
        x_fp = (self.scale) * (x_q.float() - self.zero_point)
        # x_fp = (self.scale) * (x_q.float())
        return x_fp


## Auxiliry functions

In [5]:
def safe_shift_l(x, shift):
    shift_left = torch.clamp(shift, min=0)
    shift_right = torch.clamp(-shift, min=0)
    return (x << shift_left) >> shift_right

def safe_shift_r(x, shift):
    shift_left = torch.clamp(shift, min=0)
    shift_right = torch.clamp(-shift, min=0)
    return (x >> shift_left) << shift_right


In [6]:
list_dist_exp = []
def build_exp_lut(nof_bits=16, LUT_SIZE=16):
  scale = 1 << (nof_bits - 1)
  exp_lut = torch.zeros(LUT_SIZE + LUT_SIZE, dtype=torch.int64)
  for i, offset in enumerate(range(-LUT_SIZE, LUT_SIZE)):
      exp_lut[i] = torch.round(torch.exp(torch.tensor(offset).float()) * scale).to(dtype=torch.int32)

  return exp_lut


def build_ln_lut(nof_bits=16, LUT_SIZE=16, eps=1e-5):
  scale = 1 << (nof_bits - 1)
  ln_lut = torch.zeros(LUT_SIZE + 1, dtype=torch.int64)
  ln_lut[0] = torch.round(torch.log(torch.tensor(0.25+eps).float()) * scale).to(dtype=torch.int32)  # TODO try to replace with 0.5
  for i, offset in enumerate(range(1, LUT_SIZE)):
      ln_lut[i] = torch.round(torch.log(torch.tensor(2 ** (offset-1) + eps).float()) * scale).to(dtype=torch.int32)

  return ln_lut


def int_approx_lower_power_of_two(x: torch.Tensor):
    # Handle zeros explicitly
    lut_idx = torch.where(
        x == 0,
        torch.tensor(0, dtype=torch.int32, device=x.device),  # return 0 for x==0
        # torch.floor(torch.log2(torch.clamp(x, min=0.25))).to(torch.int32)
        x.log2().floor().to(torch.int32)
    )
    return lut_idx


def TylorNLog(in_x, ev_point=-1, nof_bits=16, iterations=3, LUT_SIZE=16, ln_lut=None, LN2=None):
      """
      Approximate natural logarithm (ln) using a Taylor series expansion around evaluation points.

      Args:
          in_x (torch.Tensor): Input tensor.
          ev_point (int): Evaluation point for the Taylor series expansion. Default is -1 (automatically determined).
          nof_bits (int): Number of bits for quantization.
          iterations (int): Number of iterations for the Taylor series expansion.
          LUT_SIZE (int): Size of the lookup table (LUT).

      Returns:
          torch.Tensor: Approximated logarithmic value using Taylor series.
      """
      dtype = torch.int32
      dtype_w = dtype = torch.int32

      scale = torch.tensor(1 << (nof_bits - 1)).to(dtype)

      # emulated lut
      lut_idx = (int_approx_lower_power_of_two(in_x >> (nof_bits - 4)) - 3).clamp_(-4, LUT_SIZE + 4)

      # k = torch.tensor(lut_idx << (nof_bits - 1)).to(dtype)
      k = (lut_idx << (nof_bits - 1)).to(dtype)
      sum_result = ((k>>1)+(k>>3)+(k>>4))

      if iterations == 0:
          return sum_result

      x_diff = (in_x - (1 << (nof_bits + lut_idx - 1))).to(dtype_w)
      div_scale = (1 << (15 - lut_idx)).to(dtype)
      x_pow = ((x_diff * div_scale) >> (15)).to(dtype)

      sum_result += x_pow

      if iterations == 1:
          return sum_result

      x_pow2 = ((x_pow * x_pow) >> (nof_bits)).to(dtype)
      sum_result -= x_pow2

      if iterations == 2:
          return sum_result

      x_pow3 = ((x_pow2 * x_pow) // (scale * 3)).to(dtype)
      sum_result += x_pow3

      return sum_result

def new_ln(xq, bits):
    aq = xq.log2().floor().int() - bits
    # compute xq >> aq if aq > 0 else xq << -aq)
    k1 = safe_shift_r(xq, aq)
    # compute 2**(-aq))*xq
    k2 = ((aq-1) << bits)
    # compute (2**(-aq))*xq + ((aq-1)*2**bits)
    k = (k1 + k2)
    # yq = ln2q * ((2**(-aq))*xq + ((aq-1)*2**bits))
    yq = (((k>>1)+(k>>3)+(k>>4)))
    return yq

def TaylorExponent(x_scale, scale, iterations=1, input_bits=16, output_bits=16,LUT_SIZE=16, exp_lut=None):
      dtype = torch.int32
      udtype = torch.int32

      # Offset in steps of 2^(input_bits-1) ~ round(x)
      offset = ((x_scale + (1 << (input_bits - 2))) >> (input_bits - 1)).to(dtype)
      offset_clamped = offset.clamp_(-LUT_SIZE, LUT_SIZE)
      # lut_index = (offset_clamped + LUT_SIZE).to(torch.int64)

      # exp_offset = exp_lut[lut_index].to(udtype)
      exp_offset = (offset_clamped.exp() * (1 << (output_bits - 1))).round().to(dtype=dtype)
      if iterations == 0:
          return exp_offset

      x_a = x_scale - (offset << (input_bits - 1))
      x_pow = x_a

      # as provided: 1+x, then optional terms
      sum_result = (x_pow + scale)

      if iterations == 1:
          return (sum_result * exp_offset).to(udtype) >> (input_bits - 1)

      # note: your x^2 uses >> input_bits
      x_pow2 = (x_pow * x_pow) >> (input_bits)
      sum_result = (sum_result + x_pow2)

      if iterations == 2:
          return (sum_result * exp_offset).to(udtype) >> (input_bits - 1)

      x_pow3 = (x_pow2 * x_pow) // ( (1 << (input_bits - 1)) * 3)  # scale = 2^(bits-1)
      sum_result = (sum_result + x_pow3)

      return (sum_result * exp_offset).to(udtype) >> (input_bits - 1)





## Linear

In [7]:
class QuantizedLinear(nn.Linear):
    def __init__(self,
                 in_features,
                 out_features,
                 bias=True,
                 nof_bits1=8,
                 nof_bits2=8,
                 qscheme=torch.per_tensor_affine,
                 is_calibrate=False,
                 is_opt_scale=False,
                 quant=True):
        """
        A quantized version of nn.Linear that uses MinMaxObserver to quantize weights, bias, and inputs.

        Args:
            in_features (int): Number of input features.
            out_features (int): Number of output features.
            bias (bool): If set to False, the layer will not learn an additive bias. Default: True.
            dtype (torch.dtype): Data type for quantization, typically qint8 or quint8.
            qscheme (torch.qscheme): Quantization scheme, typically per_tensor_affine.
            nof_bits (int): Number of bits for quantization.
            is_calibrate (bool): If True, the observer will update min/max values but not quantize the input.
        """
        super(QuantizedLinear, self).__init__(in_features, out_features, bias)

        self.nof_bits1 = nof_bits1
        self.nof_bits2 = nof_bits2
        self.nof_bits_b = nof_bits1 + nof_bits2

        # Initialize observers for inputs, weights, and bias
        self.in_obs = MinMaxObserver(qscheme=torch.per_tensor_affine, nof_bits=nof_bits1, is_calibrate=True, name="linear")
        self.w_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric, nof_bits=nof_bits2, is_calibrate=True, name="linear")
        self.b_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric, nof_bits=self.nof_bits_b, is_calibrate=True, name="linear")

        self.is_calibrate = is_calibrate
        self.is_opt_scale = is_opt_scale
        self.is_weights_quantized = False  # Track whether weights have been quantized
        self.quant = quant  # Control whether to perform quantized operations

    def quantize_weights_and_bias(self):
        """
        Quantizes weights and bias if they have not been quantized yet.
        Sets scale and zero-point for weights and bias, and stores quantized values.
        """
        if not self.is_weights_quantized:
            # Quantize the weights
            self.w_obs(self.weight)
            self.scale_weight, self.zero_point_weight = self.w_obs.calculate_qparams()
            self.weight_integer = self.w_obs.quantizer(self.weight) - self.zero_point_weight

            # Quantize the bias if it exists
            if self.bias is not None:
                # Calculate bias quantization scale as the product of input and weight scales
                self.b_obs.scale = self.in_obs.scale * self.w_obs.scale
                self.b_obs.zero_point = 0  # Bias zero-point is typically set to 0
                self.bias_integer = self.b_obs.quantizer(self.bias)
            else:
                # Set scale and zero-point for cases without a bias
                self.b_obs.scale = self.in_obs.scale * self.w_obs.scale
                self.b_obs.zero_point = 0
                self.bias_integer = None

            # Mark weights and bias as quantized
            self.is_weights_quantized = True

    def forward_pass(self, x):
        """
        Forward pass with quantization:
        - Quantizes weights and bias if not already done.
        - Quantizes the input.
        - Performs linear transformation with quantized weights and bias.
        - Dequantizes the output before returning.
        """
        # Ensure weights and bias are quantized
        self.quantize_weights_and_bias()

        # Quantize the input
        self.scale_input, self.zero_point_input = self.in_obs.calculate_qparams()
        _x_q = self.in_obs.quantizer(x)  # Quantize the input
        x_q = _x_q - self.zero_point_input.int()  # Adjust input by zero point

        # Integer linear transformation using quantized weights and bias
        if self.bias is not None:
            output_q = F.linear(
                x_q.float(),
                weight=self.weight_integer.float(),
                bias=self.bias_integer.float()
            )
        else:
            output_q = F.linear(
                x_q.float(),
                weight=self.weight_integer.float()
            )

        # Dequantize the output
        dq_output = self.b_obs.dequantizer(output_q)
        return dq_output

    def float_forward_pass(self, x):
        return F.linear(
            x,
            self.weight,
            self.bias
        )

    def forward(self, x):
        """
        Forward pass through the quantized linear layer.
        If is_calibrate is True, update the min/max values of the input observer without performing quantization.
        Otherwise, quantize the input, weights, and bias, perform integer-based linear operation, and dequantize the result.

        Args:
            x (torch.Tensor): Input tensor to the linear layer.

        Returns:
            torch.Tensor: Dequantized output after quantized linear transformation.
        """
        # Calibration mode: update the input observer and return the input tensor as-is
        if self.is_calibrate:
            self.in_obs(x)
            return self.float_forward_pass(x)

        if self.is_opt_scale:
            self.opt_input = x
            return self.float_forward_pass(x)

        # Apply the linear transformation in the quantized domain using integer arithmetic
        if self.quant:
            return self.forward_pass(x)
        else:
            return self.float_forward_pass(x)

    def set_calibration_flag(self):
        self.is_calibrate = True

    def unset_calibration_flag(self):
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input
        self.is_opt_scale = False



## Matmul (two element)

In [8]:
class QuantizedMatmul(nn.Module):
    """
    A quantized matrix multiplication layer using integer-only arithmetic for efficiency.
    This class uses MinMaxObserver to quantize inputs and perform matrix multiplication in the quantized domain.
    """

    def __init__(self,
                 is_calibrate=False,
                 in1_bits=8,
                 in2_bits=8,
                 is_opt_scale=False,
                 quant=True):
        """
        Initializes the quantized matrix multiplication layer.

        Args:
            bias (bool): If set to False, the layer will not use bias. Default: True.
            is_calibrate (bool): Flag for enabling calibration mode to update observers.
        """
        super(QuantizedMatmul, self).__init__()

        self.in1_bits=in1_bits
        self.in2_bits=in2_bits
        self.is_calibrate = is_calibrate
        self.quant = quant  # Whether to use quantized operations
        self.is_weights_quantized = False
        self.is_opt_scale = is_opt_scale

        # Initialize observers for two input tensors and the output
        self.in1_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                      nof_bits=self.in1_bits,
                                      is_calibrate=is_calibrate,
                                      name="MatMul")
        self.in2_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                      nof_bits=self.in2_bits,
                                      is_calibrate=is_calibrate,
                                      name="MatMul")
        self.out_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric,
                                      nof_bits=(self.in1_bits+self.in2_bits),
                                      is_calibrate=is_calibrate,
                                      name="MatMul")

    def quantize_weights_and_bias(self):
        """
        Quantizes weights and bias if they have not been quantized yet.
        Sets scale and zero-point for weights and bias, and stores quantized values.
        """
        if not self.is_weights_quantized:

            # Calculate bias quantization scale as the product of input and weight scales
            self.out_obs.scale = self.in1_obs.scale * self.in2_obs.scale
            self.out_obs.zero_point = 0  # Bias zero-point is typically set to 0

            # Mark weights and bias as quantized
            self.is_weights_quantized = True


    def forward_pass(self, x1, x2):
        self.quantize_weights_and_bias()

        # Quantize the inputs
        scale_input1, zero_point_input1 = self.in1_obs.calculate_qparams()
        scale_input2, zero_point_input2 = self.in2_obs.calculate_qparams()

        # Quantize the inputs based on their scales and zero points
        x_q1 = self.in1_obs.quantizer(x1)
        x_q2 = self.in2_obs.quantizer(x2)

        # Perform quantized matrix multiplication
        output_q = (x_q1 - zero_point_input1) @ (x_q2 - zero_point_input2)

        # Dequantize the output
        dq_output = self.out_obs.dequantizer(output_q)
        return dq_output

    def float_forward_pass(self, x1, x2):
        return x1 @ x2

    def forward(self, x1, x2):
        """
        Forward pass for matrix multiplication.

        Args:
            x1 (torch.Tensor): The first input tensor.
            x2 (torch.Tensor): The second input tensor.

        Returns:
            torch.Tensor: Result of matrix multiplication, either quantized or floating-point.
        """
        # Calibration mode: update observers with input min/max values
        if self.is_calibrate:
            self.in1_obs(x1)
            self.in2_obs(x2)
            return self.float_forward_pass(x1, x2)  # Return the floating-point result during calibration

        if self.is_opt_scale:
            self.opt_input1 = x1
            self.opt_input2 = x2
            return self.float_forward_pass(x1, x2)  # Return the floating-point result during calibration

        # Quantized matrix multiplication
        if self.quant:
            return self.forward_pass(x1, x2)
        else:
            # Standard floating-point matrix multiplication
            return self.float_forward_pass(x1, x2)

    def set_calibration_flag(self):
        """Enable calibration mode to update observers."""
        self.is_calibrate = True

    def unset_calibration_flag(self):
        """Disable calibration mode."""
        self.is_calibrate = False


    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input1
        del self.opt_input2
        self.is_opt_scale = False


## Softmax

In [9]:
sf_layer_idx = 0
class IntSoftmaxTS(nn.Module):
    """
    Quantized Softmax with a combination of Lookup Table (LUT) approximation and Taylor Series expansion.
    This class uses integer-only arithmetic to perform the softmax function with quantized inputs.
    """

    def __init__(self,
                 quant=False,
                 is_calibrate=False,
                 nof_bits=16,
                 LUT_SIZE=16,
                 is_opt_scale=False,
                 eps=1e-5,
                 dim=-1,
                 iterations=1,
                 ts_ln=TylorNLog,
                 int_exp_scaled=TaylorExponent):
        super(IntSoftmaxTS, self).__init__()

        self.LUT_SIZE = LUT_SIZE
        self.nof_bits = nof_bits
        self.iterations = iterations
        self.dim = dim
        self.quant = quant
        self.is_calibrate = is_calibrate
        self.is_opt_scale = is_opt_scale
        self.eps = eps

        scale = 1 << (self.nof_bits - 1)
        self.div3 = int(round(scale / 3))

        self.ts_ln = ts_ln
        self.int_exp_scaled = int_exp_scaled

        self.input_bits = self.nof_bits - 4
        self.output_bits = self.nof_bits + 1
        self.stats = dict()
        self.stats[f'softmax'] = []

        self.ln2 = (torch.tensor(2).log() * 2 ** (self.output_bits-1)).round().to(torch.int32)

        self.in_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric,
                                     nof_bits=self.nof_bits-2,
                                     is_calibrate=is_calibrate,
                                     name="Softmax")

        self.exp_lut = []
        self.ln_lut =  []

        # self.exp_lut = build_exp_lut(nof_bits=self.output_bits, LUT_SIZE=self.LUT_SIZE)
        # self.ln_lut = build_ln_lut(nof_bits=self.output_bits, LUT_SIZE=self.LUT_SIZE, eps=self.eps)

        # if torch.cuda.is_available():
        #     self.exp_lut = self.exp_lut.cuda()
        #     self.ln_lut = self.ln_lut.cuda()



    def int_softmax(self, x):

        dtype = torch.int32
        udtype = torch.int32

        self.scale_input, self.zero_point_input = self.in_obs.calculate_qparams()
        _x_q = self.in_obs.quantizer(x)  # Quantize the input
        x_q = _x_q  # Adjust input by zero point

        # reference
        # x_int = x - x.max(dim=self.dim, keepdim=True).values

        # rounding float
        # x_round = x.round()
        # x_int = x_round - x_round.max(dim=self.dim, keepdim=True).values

        # quantized 4 bits
        x_int = self.in_obs.dequantizer(x_q - x_q.max(dim=self.dim, keepdim=True).values).round()

        spacial_scale = 1 << (self.input_bits - 1)
        x_scale = (x_int * spacial_scale).floor().to(dtype)

        try:
            exp_int = self.int_exp_scaled(x_scale,
                                          spacial_scale,
                                          input_bits=self.input_bits,
                                          output_bits=self.output_bits,
                                          LUT_SIZE=self.LUT_SIZE,
                                          exp_lut=self.exp_lut,
                                          iterations=1)  # TODO test zero iterations
                                          # iterations=self.iterations)
        except RuntimeError as err:
            print("[EXP 1] : TEST")
            print(x_int)
            raise err
        exp_int_sum = exp_int.sum(dim=-1, keepdim=True)

        ln_sum = self.ts_ln(exp_int_sum,
                            iterations=self.iterations+1,
                            nof_bits=self.output_bits,
                            ln_lut=self.ln_lut,
                            LN2=self.ln2)

        spacial_scale = 1 << (self.output_bits - 1)

        try:
            ln_mul = self.int_exp_scaled(-ln_sum.int(),
                                        spacial_scale,
                                        input_bits=self.output_bits,
                                        output_bits=self.output_bits,
                                        LUT_SIZE=self.LUT_SIZE,
                                        exp_lut=self.exp_lut,
                                        iterations=self.iterations)
        except RuntimeError as err:
            print("[EXP 2] : TEST")
            print(x_int)
            raise err

        sf_values = ln_mul * exp_int
        return sf_values

    def foward_pass(self, x):
        sf_values = self.int_softmax(x)
        deq_softmax = sf_values / (1 << ((self.output_bits - 1) + (self.output_bits - 1)))
        return deq_softmax

    def float_forward_pass(self, x):
        return x.softmax(dim=self.dim)


    def forward(self, x):
        if self.is_calibrate:
            self.in_obs(x);  return x.softmax(dim=self.dim)

        if self.is_opt_scale:
            self.opt_input = x;  return x.softmax(dim=self.dim)

        # self.stats[f'softmax'].append(x)

        if self.quant:
            return self.foward_pass(x)
        else:
            return self.float_forward_pass(x)

    def set_calibration_flag(self):
        self.is_calibrate = True

    def unset_calibration_flag(self):
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input
        self.is_opt_scale = False


## Hadamard mul (Mul by Element)

In [10]:
class qHadamardProd(nn.Module):
    """
    A quantized matrix multiplication layer using integer-only arithmetic for efficiency.
    This class uses MinMaxObserver to quantize inputs and perform matrix multiplication in the quantized domain.
    """

    def __init__(self,
                 nof_bits1=16,
                 nof_bits2=16,
                 quant=True,
                 is_calibrate=False):
        """
        Initializes the quantized matrix multiplication layer.

        Args:
            bias (bool): If set to False, the layer will not use bias. Default: True.
            is_calibrate (bool): Flag for enabling calibration mode to update observers.
        """
        super(qHadamardProd, self).__init__()
        self.nof_bits1=nof_bits1
        self.nof_bits2=nof_bits2
        self.nof_bits_o = self.nof_bits1 + self.nof_bits2

        # Initialize observers for two input tensors and the output
        self.in1_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                      nof_bits=self.nof_bits1,
                                      is_calibrate=is_calibrate,
                                      name="hadamard")
        self.in2_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                      nof_bits=self.nof_bits2,
                                      is_calibrate=is_calibrate,
                                      name="hadamard")
        self.out_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric,
                                      nof_bits=self.nof_bits_o,
                                      is_calibrate=is_calibrate,
                                      name="hadamard")

        self.is_calibrate = is_calibrate
        self.quant = quant  # Whether to use quantized operations
        self.is_out_scaled = False

    def set_out_scale(self):
        """
        Quantizes weights and bias if they have not been quantized yet.
        Sets scale and zero-point for weights and bias, and stores quantized values.
        """
        if not self.is_out_scaled:

            # Calculate bias quantization scale as the product of input and weight scales
            self.out_obs.scale = self.in1_obs.scale * self.in2_obs.scale
            self.out_obs.zero_point = 0  # Bias zero-point is typically set to 0

            # Mark weights and bias as quantized
            self.is_out_scaled = True

    def float_forward_pass(self, x1, x2):
        return x1 * x2

    def forward_pass(self, x1, x2):
        self.set_out_scale()

        # Quantize the inputs
        _, zero_point_input1 = self.in1_obs.calculate_qparams()
        _, zero_point_input2 = self.in2_obs.calculate_qparams()

        # Quantize the inputs based on their scales and zero points
        x_q1 = self.in1_obs.quantizer(x1)
        x_q2 = self.in2_obs.quantizer(x2)

        # Perform quantized matrix multiplication
        output_q = (x_q1 - zero_point_input1) * (x_q2 - zero_point_input2)

        # Dequantize the output
        dq_output = self.out_obs.dequantizer(output_q)
        return dq_output

    def forward(self, x1, x2):
        """
        Forward pass for matrix multiplication.

        Args:
            x1 (torch.Tensor): The first input tensor.
            x2 (torch.Tensor): The second input tensor.

        Returns:
            torch.Tensor: Result of matrix multiplication, either quantized or floating-point.
        """
        # Calibration mode: update observers with input min/max values
        if self.is_calibrate:
            self.in1_obs(x1)
            self.in2_obs(x2)
            return self.float_forward_pass(x1, x2)  # Return the floating-point result during calibration

        # Quantized matrix multiplication
        if self.quant:
            return self.forward_pass(x1, x2)
        else:
            # Standard floating-point matrix multiplication
            return self.float_forward_pass(x1, x2)


    def set_calibration_flag(self):
        """Enable calibration mode to update observers."""
        self.is_calibrate = True

    def unset_calibration_flag(self):
        """Disable calibration mode."""
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        # del self.opt_input
        self.is_opt_scale = False


## GeLU

In [11]:
from transformers.activations import ACT2FN

class IntGeluTS(nn.Module):
    """
    Quantized GELU using a combination of Lookup Table (LUT) approximation and Taylor Series expansion.
    This class uses integer-only arithmetic to perform the GELU function with quantized inputs.
    """

    def __init__(self,
                 quant=False,
                 LUT_SIZE=16,
                 nof_bits=16,
                 eps=1e-5,
                 gelu_scale=1.702,
                 is_opt_scale=False,
                 is_calibrate=False,
                 ts_ln=TylorNLog,
                 int_exp_scaled=TaylorExponent):
        super(IntGeluTS, self).__init__()

        self.LUT_SIZE = LUT_SIZE
        self.nof_bits = nof_bits
        self.iterations = 2
        self.is_opt_scale = is_opt_scale
        self.is_weights_quantized = True
        self.gelu_scale = gelu_scale
        self.ts_ln = ts_ln
        self.int_exp_scaled = int_exp_scaled
        self.eps = eps

        self.input_bits = self.nof_bits - 4
        self.output_bits = self.nof_bits + 1

        # self.ln2 = (torch.tensor(2).log() * 2 ** (self.output_bits-1)).round().to(torch.int32)

        self.quant = quant
        self.is_calibrate = is_calibrate

        self.had_mul = qHadamardProd(nof_bits1=nof_bits,
                                     nof_bits2=nof_bits,
                                     quant=quant)

        self.in_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                     nof_bits=nof_bits,
                                     is_calibrate=True,
                                     name="geLU")

        self.k_values = torch.tensor([
            2.834596, 2.338217, 1.978175, 1.754823, 1.642511,
            1.642511, 1.754823, 1.978175, 2.338217, 2.834596
        ]).cuda()

        self.exp_lut = []
        self.ln_lut =  []
        self.fp_gelu = ACT2FN[config.hidden_act]   # pass in config or string

        # self.exp_lut = build_exp_lut(nof_bits=self.output_bits, LUT_SIZE=self.LUT_SIZE)
        # self.ln_lut = build_ln_lut(nof_bits=self.output_bits, LUT_SIZE=self.LUT_SIZE, eps=self.eps)

        # if torch.cuda.is_available():
        #     self.exp_lut = self.exp_lut.cuda()
        #     self.ln_lut = self.ln_lut.cuda()
        #     self.k_values = self.k_values.cuda()

    def int_sigmoid(self, x, nof_bits=16, iterations=2, gelu_scale=1.702, LUT_SIZE=-1):
        input_bits = nof_bits - 4
        output_bits = nof_bits

        x_sig = x * gelu_scale
        x_max = torch.clamp(x_sig, min=0)
        x_int = x_sig - x_max

        spacial_scale = 1 << (input_bits - 1)
        x_scale = (x_int * spacial_scale).floor().to(dtype=torch.int32)
        x_max_scale = -(x_max * spacial_scale).floor().to(dtype=torch.int32)

        exp_int = TaylorExponent(x_scale,
                              spacial_scale,
                              input_bits=input_bits,
                              output_bits=output_bits,
                              LUT_SIZE=LUT_SIZE,
                              exp_lut=[],
                              iterations=0)
                              # iterations=iterations)

        exp_zero = TaylorExponent(x_max_scale,
                              spacial_scale,
                              input_bits=input_bits,
                              output_bits=output_bits,
                              LUT_SIZE=LUT_SIZE,
                              exp_lut=[],
                              iterations=iterations)

        exp_int_sum = exp_int + exp_zero

        return exp_int, exp_int_sum

    def get_k_value(self, x):
        x = torch.clamp(x, -5, 5)
        x_shifted = x + 5
        indices = torch.clamp(x_shifted.floor().long(), 0, 9)
        return self.k_values[indices]

    def forward_pass(self, x):
        # alpha = self.get_k_value(x)
        alpha = 1.702  # self.get_k_value(x)

        exp_int, exp_int_sum = self.int_sigmoid(x, self.nof_bits, self.iterations, alpha, self.LUT_SIZE)

        ln_sum = self.ts_ln(exp_int_sum,
                            iterations=self.iterations+1,
                            nof_bits=self.output_bits,
                            ln_lut=self.ln_lut,
                            LN2=None)

        spacial_scale_out = 1 << (self.output_bits - 1)

        ln_mul = self.int_exp_scaled(-ln_sum,
                                     spacial_scale_out,
                                     input_bits=self.output_bits,
                                    output_bits=self.output_bits,
                                    LUT_SIZE=self.LUT_SIZE,
                                    exp_lut=[],
                                    iterations=self.iterations)

        q_sigmoid = ln_mul * exp_int
        deq_sigmoid = q_sigmoid / (1 << ((self.output_bits - 1) + (self.output_bits - 1)))

        return self.had_mul(x, deq_sigmoid)

    def float_forward_pass(self, x):
        normal = torch.distributions.Normal(0.0, 1.0)
        sig = normal.cdf(x) # collect data for other observers
        _ = self.had_mul(x, sig) # collect data for other observers
        # return self.had_mul(x, sig)
        return self.fp_gelu(x)



    def quantize_weights_and_bias(self):
        if not self.is_weights_quantized:
            self.is_weights_quantized = True

    def forward(self, x):
        if self.is_calibrate:
            self.in_obs(x)
            return self.float_forward_pass(x)

        if self.is_opt_scale:
            self.opt_input = x
            return self.float_forward_pass(x)

        return self.forward_pass(x) if self.quant else self.float_forward_pass(x)

    def set_calibration_flag(self):
        self.is_calibrate = True

    def unset_calibration_flag(self):
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input
        self.is_opt_scale = False


## Layer norm

In [12]:
class QLayerNorm(nn.LayerNorm):
    """
    A subclass of PyTorch's LayerNorm with quantization support.

    Args:
        normalized_shape: Input shape from an expected input of size (N, *), where * means any number of additional dimensions.
        eps: A value added to the denominator for numerical stability.
        elementwise_affine: When set to True, this module has learnable per-element affine parameters.
    """
    def __init__(self,
                 normalized_shape,
                 eps=1e-5,
                 in1_bits=16,
                 in2_bits=16,
                 LUT_SIZE=16,
                 quant=False,
                 elementwise_affine=True,
                 is_opt_scale=False,
                 is_calibrate=False,
                 ts_ln=TylorNLog,
                 int_exp_scaled=TaylorExponent,
                 iterations=3):

        super(QLayerNorm, self).__init__(normalized_shape, eps, elementwise_affine)
        self.ts_ln = ts_ln
        self.int_exp_scaled = int_exp_scaled

        self.in1_bits = in1_bits
        self.in2_bits = in2_bits-1
        self.is_opt_scale = is_opt_scale
        self.is_calibrate = is_calibrate
        self.LUT_SIZE = LUT_SIZE
        self.quant = quant  # Control whether to perform quantized operations
        self.is_weights_quantized = False  # Track whether weights have been quantized
        self.iterations = iterations
        self.input_bits1 = self.in1_bits
        self.output_bits1 = self.in1_bits

        self.in_obs_normalize = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                     nof_bits=self.input_bits1,
                                     is_calibrate=True,
                                     name="lnorm")

        # Initialize observers for inputs, weights, and self.bias
        self.in_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                     nof_bits=self.input_bits1,
                                     is_calibrate=True,
                                     name="lnorm")
        self.w_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                    nof_bits=self.in2_bits,
                                    is_calibrate=True,
                                    name="lnorm")

        if self.bias is not None:
            self.b_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric,
                                      nof_bits=self.in1_bits+self.in2_bits,
                                      is_calibrate=True,
                                      name="lnorm")
        else:
            self.b_obs = None

        self.output_bits1 = self.in1_bits
        self.exp_lut = build_exp_lut(nof_bits=self.output_bits1, LUT_SIZE=self.LUT_SIZE)
        self.ln_lut = build_ln_lut(nof_bits=self.output_bits1, LUT_SIZE=self.LUT_SIZE, eps=self.eps)
        self.ln2 = (torch.tensor(2).log() * 2 ** (self.output_bits1-1)).round().to(torch.int32)
        self.stats = dict()

        self.stats['scale'] = []
        self.stats['ref'] = []
        self.stats['var'] = torch.tensor([]).cuda()
        # self.sample_counter = 0
        self.sample_counter = 32 # do not sample

        if torch.cuda.is_available():
            self.exp_lut = self.exp_lut.cuda()
            self.ln_lut = self.ln_lut.cuda()

    # Define ts_ln function
    def scaled_ln16(self, in_x, nof_bits=8):
        scale_factor = 1 << (self.input_bits1 - 1)
        scaled_input = (in_x * scale_factor).floor().to(dtype=torch.int32)

        # log_factor = torch.where(
        #     scaled_input >> (self.output_bits1 - 1) == 0,
        #     2,
        #     0
        # )


        # ln_sum = self.ts_ln(scaled_input << (log_factor << 1),
        #                     iterations=3,
        #                     nof_bits=self.output_bits1,
        #                     ln_lut=self.ln_lut,
        #                     LN2=self.ln2) >> 1


        # out = self.int_exp_scaled(-ln_sum,
        #                           scale_factor,
        #                           input_bits=self.output_bits1,
        #                           output_bits=self.output_bits1,
        #                           LUT_SIZE=self.LUT_SIZE,
        #                           exp_lut=self.exp_lut,
        #                           iterations=2) << log_factor

        ln_sum = new_ln(scaled_input, self.output_bits1-1)

        out = TaylorExponent(-ln_sum>>1,
                             scale_factor,
                             iterations=2,
                             input_bits=self.output_bits1,
                             output_bits=self.output_bits1)

        return out

    def float_foward_pass_normalize(self, x):
        x_mean = x.float().mean(dim=-1, keepdim=True)
        x_var = x.float().var(dim=-1, unbiased=False, keepdim=True)
        x_div = x_var.sqrt()
        normed_x = (x - x_mean) / x_div
        return normed_x

    def quantize_weights_and_bias(self):
        """
        Quantizes weights and bias if they have not been quantized yet.
        Sets scale and zero-point for weights and bias, and stores quantized values.
        """
        if not self.is_weights_quantized:
            # Quantize the weights
            self.w_obs(self.weight)
            self.scale_weight, self.zero_point_weight = self.w_obs.calculate_qparams()
            self.weight_integer = self.w_obs.quantizer(self.weight)
            # print(f"[DEBUG] : self.zero_point_weight: {self.zero_point_weight}, min val: {self.w_obs.min_val}, max val: {self.w_obs.max_val}, scale: {self.scale_weight}")
            # Quantize the bias if it exists
            if self.bias is not None:
                # Calculate bias quantization scale as the product of input and weight scales
                self.b_obs.scale = self.in_obs.scale * self.w_obs.scale
                self.b_obs.zero_point = 0  # bias zero-point is typically set to 0
                self.bias_integer = self.b_obs.quantizer(self.bias)
            else:
                # Set scale and zero-point for cases without a bias
                self.b_obs.scale = self.in_obs.scale * self.w_obs.scale
                self.b_obs.zero_point = 0
                self.bias_integer = None

            # Mark weights and bias as quantized
            self.is_weights_quantized = True

    def betta_gamma_forward_pass(self, x):
        """
        Forward pass with quantization:
        - Quantizes weights and bias if not already done.
        - Quantizes the input.
        - Performs linear transformation with quantized weights and bias.
        - Dequantizes the output before returning.
        """

        # Ensure weights and bias are quantized
        self.quantize_weights_and_bias()

        # Quantize the input
        _ , self.zero_point_input = self.in_obs.calculate_qparams()
        # _ , self.zero_point_weight = self.w_obs.calculate_qparams()
        x_q = self.in_obs.quantizer(x)  # Quantize the input
        x_q = x_q - self.zero_point_input.int()  # Adjust input by zero point

        # Integer linear transformation using quantized weights and bias
        if self.bias is not None:
            output_q = x_q * (self.weight_integer - self.zero_point_weight.int()) + self.bias_integer
        else:
            output_q = x_q * (self.weight_integer - self.zero_point_weight.int())

        # print(self.w_obs.dequantizer(self.weight_integer + self.zero_point_weight.int()))
        # print("-----------------")
        # print(self.weight)
        # print("=================")

        # Dequantize the output
        dq_output = self.b_obs.dequantizer(output_q)
        return dq_output

    def normalize(self, x):
        self.scale_input_normed , self.zero_point_input_normed = self.in_obs_normalize.calculate_qparams()

        x_q = self.in_obs_normalize.quantizer(x)
        x_q = x_q - self.zero_point_input_normed

        xq_mean = x_q.float().mean(dim=-1, keepdim=True).long()
        xq_var = x_q.float().var(dim=-1, unbiased=False, keepdim=True)
        x_var = ((self.scale_input_normed) ** 2) * (xq_var.float())

        ln_mul = self.scaled_ln16(x_var, nof_bits=self.in1_bits)

        q_mean_val = (x_q - xq_mean)
        x_dq = ln_mul * q_mean_val * ((self.scale_input_normed) / 2**(self.in1_bits-1))

        # x_mean = x.float().mean(dim=-1, keepdim=True)
        # x_var = x.float().var(dim=-1, unbiased=False, keepdim=True)
        # x_div = x_var.sqrt()

        return x_dq

    def float_betta_gamma_forward_pass(self, x):
        return x * self.weight + self.bias

    def forward_pass(self, x):
        mean_val = self.normalize(x)
        # mean_val = self.float_foward_pass_normalize(x)
        dq_output = self.betta_gamma_forward_pass(mean_val)
        # dq_output = self.float_betta_gamma_forward_pass(mean_val)
        # self.stats['var'].append(dq_output)
        # self.stats['ref'].append(self.float_betta_gamma_forward_pass(mean_val))

        return dq_output

    def float_forward_pass(self, x):
        # if self.sample_counter < 32:
        #     self.stats['var'] = torch.cat((self.stats['var'], x.var(dim=-1, unbiased=False, keepdim=True)), dim=-1)
        #     self.sample_counter += 1

        return F.layer_norm(x,
                            self.normalized_shape,
                            weight=self.weight,
                            bias=self.bias,
                            eps=self.eps
        )

    def forward(self, x):

        # Calibration mode: update the input observer and return the input tensor as-is
        if self.is_calibrate:
            self.in_obs_normalize(x)
            normed_x = self.float_foward_pass_normalize(x)
            _ = self.in_obs(normed_x)
            return self.float_forward_pass(x)

        if self.is_opt_scale:
            self.opt_input = x
            return self.float_forward_pass(x)

        if self.quant:
            dq_output = self.forward_pass(x)
            return dq_output
        else:
            return self.float_forward_pass(x)

    def set_calibration_flag(self):
        """Enable calibration mode to update observers."""
        self.is_calibrate = True

    def unset_calibration_flag(self):
        """Disable calibration mode."""
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input
        self.is_opt_scale = False


# Bert model

## Base Bert Model

In [13]:
# Quantization parameters class
class QauntParams(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = False
        self.nof_bits_linear1 = 8
        self.nof_bits_linear2 = 8
        self.nof_bits_gelu    = 8
        self.nof_bits_softmax = 8
        self.lut_size_softmax = 7
        self.nof_bits_lnorm1 = 12
        self.nof_bits_lnorm2 = 4
        self.nof_bits_matmul1 = 8
        self.nof_bits_matmul2 = 8


    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                print("I WAS HERE :)")
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.unset_scale_opt()


In [14]:
class QBertPreTrainedModel(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.q_module_list = []
        self.quant = False
        self.nof_bits_linear1 = 8
        self.nof_bits_linear2 = 8
        self.nof_bits_gelu    = 16
        self.nof_bits_softmax = 8
        self.lut_size_softmax = 8
        self.nof_bits_lnorm1 = 16
        self.nof_bits_lnorm2 = 16

    def set_q_module_list(self, q_module_list):
        self.q_module_list = q_module_list

    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.unset_scale_opt()


In [15]:
class CustomBertPooler(QauntParams):
    def __init__(self, config):
        super().__init__()
        # self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dense = QuantizedLinear(config.hidden_size,
                                    config.hidden_size,
                                    nof_bits1=self.nof_bits_linear1,
                                    nof_bits2=self.nof_bits_linear2,
                                    quant=self.quant)
        self.activation = nn.Tanh()

    def forward(self, hidden_states):
        # We "pool" the model by simply taking the hidden state corresponding
        # to the first token.
        first_token_tensor = hidden_states[:, 0]
        pooled_output = self.dense(first_token_tensor)
        pooled_output = self.activation(pooled_output)
        return pooled_output


In [16]:
class CustomBertSelfOutput(QauntParams):
    def __init__(self, config):
        super().__init__()
        # self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dense = QuantizedLinear(config.hidden_size,
                            config.hidden_size,
                            nof_bits1=self.nof_bits_linear1,
                            nof_bits2=self.nof_bits_linear2,
                            quant=self.quant)

        # self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.LayerNorm = QLayerNorm(config.hidden_size,
                                    in1_bits=self.nof_bits_lnorm1,
                                    in2_bits=self.nof_bits_lnorm2,
                                    eps=config.layer_norm_eps,
                                    quant=self.quant)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


In [17]:
class CustomBertSelfAttention(QauntParams):
    def __init__(self, config):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                "The hidden size (%d) is not a multiple of the number of attention "
                "heads (%d)" % (config.hidden_size, config.num_attention_heads)
            )

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        # self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.query = QuantizedLinear(config.hidden_size,
                                    self.all_head_size,
                                    nof_bits1=self.nof_bits_linear1,
                                    nof_bits2=self.nof_bits_linear2,
                                    quant=self.quant)

        # self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = QuantizedLinear(config.hidden_size,
                                    self.all_head_size,
                                    nof_bits1=self.nof_bits_linear1,
                                    nof_bits2=self.nof_bits_linear2,
                                    quant=self.quant)

        # self.value = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = QuantizedLinear(config.hidden_size,
                                    self.all_head_size,
                                    nof_bits1=self.nof_bits_linear1,
                                    nof_bits2=self.nof_bits_linear2,
                                    quant=self.quant)

        self.mat_mul_qk = QuantizedMatmul(in1_bits=self.nof_bits_matmul1,
                                          in2_bits=self.nof_bits_matmul2,
                                          quant=self.quant)

        self.sf = IntSoftmaxTS(nof_bits=self.nof_bits_softmax,
                               LUT_SIZE=self.lut_size_softmax,
                               dim=-1,
                               quant=self.quant)

        self.mat_mul_pv = QuantizedMatmul(in1_bits=self.nof_bits_matmul1,
                                          in2_bits=self.nof_bits_matmul2,
                                          quant=self.quant)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self.stats = dict()
        self.stats[f'attn'] = []

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        # print("Input shape to transpose_for_scores:", x.shape)
        return x.permute(0, 2, 1, 3)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        output_attentions=False,
    ):
        mixed_query_layer = self.query(hidden_states)

        # If this is instantiated as a cross-attention module, the keys
        # and values come from an encoder; the attention mask needs to be
        # such that the encoder's padding tokens are not attended to.
        if encoder_hidden_states is not None:
            mixed_key_layer = self.key(encoder_hidden_states)
            mixed_value_layer = self.value(encoder_hidden_states)
            attention_mask = encoder_attention_mask
        else:
            mixed_key_layer = self.key(hidden_states)
            mixed_value_layer = self.value(hidden_states)

        query_layer = self.transpose_for_scores(mixed_query_layer)
        key_layer = self.transpose_for_scores(mixed_key_layer)
        value_layer = self.transpose_for_scores(mixed_value_layer)

        # Take the dot product between "query" and "key" to get the raw attention scores.
        # attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = self.mat_mul_qk(query_layer, key_layer.transpose(-1, -2))

        attention_scores = attention_scores / math.sqrt(self.attention_head_size)
        if attention_mask is not None:
            # Apply the attention mask is (precomputed for all layers in BertModel forward() function)
            attention_scores = attention_scores + attention_mask

        # Normalize the attention scores to probabilities.
        # attention_probs = nn.Softmax(dim=-1)(attention_scores)
        attention_probs = self.sf(attention_scores)


        # This is actually dropping out entire tokens to attend to, which might
        # seem a bit unusual, but is taken from the original Transformer paper.
        attention_probs = self.dropout(attention_probs)

        # Mask heads if we want to
        if head_mask is not None:
            attention_probs = attention_probs * head_mask

        # context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = self.mat_mul_pv(attention_probs, value_layer)

        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)

        outputs = (context_layer, attention_probs) if output_attentions else (context_layer,)

        # self.stats['attn'].append(attention_probs)

        return outputs


In [18]:
class CustomBertAttention(QauntParams):
    def __init__(self, config):
        super().__init__()
        self.self = CustomBertSelfAttention(config)
        self.output = CustomBertSelfOutput(config)
        self.pruned_heads = set()

    def prune_heads(self, heads):
        if len(heads) == 0:
            return
        heads, index = find_pruneable_heads_and_indices(
            heads, self.self.num_attention_heads, self.self.attention_head_size, self.pruned_heads
        )

        # Prune linear layers
        self.self.query = prune_linear_layer(self.self.query, index)
        self.self.key = prune_linear_layer(self.self.key, index)
        self.self.value = prune_linear_layer(self.self.value, index)
        self.output.dense = prune_linear_layer(self.output.dense, index, dim=1)

        # Update hyper params and store pruned heads
        self.self.num_attention_heads = self.self.num_attention_heads - len(heads)
        self.self.all_head_size = self.self.attention_head_size * self.self.num_attention_heads
        self.pruned_heads = self.pruned_heads.union(heads)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        output_attentions=False,
    ):
        self_outputs = self.self(
            hidden_states,
            attention_mask,
            head_mask,
            encoder_hidden_states,
            encoder_attention_mask,
            output_attentions,
        )
        attention_output = self.output(self_outputs[0], hidden_states)
        outputs = (attention_output,) + self_outputs[1:]  # add attentions if we output them
        return outputs



In [19]:
class CustomBertIntermediate(QauntParams):
    def __init__(self, config):
        super().__init__()
        # self.dense = nn.Linear(config.hidden_size, config.intermediate_size)
        self.dense = QuantizedLinear(config.hidden_size,
                                    config.intermediate_size,
                                    nof_bits1=self.nof_bits_linear1,
                                    nof_bits2=self.nof_bits_linear2,
                                    quant=self.quant)

        if isinstance(config.hidden_act, str):
            # self.intermediate_act_fn = ACT2FN[config.hidden_act]
            self.intermediate_act_fn = IntGeluTS(quant=self.quant,
                                                LUT_SIZE=16,
                                                nof_bits=8)
        else:
            self.intermediate_act_fn = config.hidden_act

    def forward(self, hidden_states):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states


class CustomBertOutput(QauntParams):
    def __init__(self, config):
        super().__init__()
        # self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dense = QuantizedLinear(config.intermediate_size,
                            config.hidden_size,
                            nof_bits1=self.nof_bits_linear1,
                            nof_bits2=self.nof_bits_linear2,
                            quant=self.quant)

        # self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.LayerNorm = QLayerNorm(config.hidden_size,
                                    in1_bits=self.nof_bits_lnorm1,
                                    in2_bits=self.nof_bits_lnorm2,
                                    eps=config.layer_norm_eps,
                                    quant=self.quant)

        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states, input_tensor):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


In [20]:
class CustomBertLayer(QauntParams):
    def __init__(self, config):
        super().__init__()
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        self.attention = CustomBertAttention(config)
        self.is_decoder = config.is_decoder
        self.add_cross_attention = config.add_cross_attention
        if self.add_cross_attention:
            assert self.is_decoder, f"{self} should be used as a decoder model if cross attention is added"
            self.crossattention = CustomBertAttention(config)
        self.intermediate = CustomBertIntermediate(config)
        self.output = CustomBertOutput(config)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        output_attentions=False,
    ):
        self_attention_outputs = self.attention(
            hidden_states,
            attention_mask,
            head_mask,
            output_attentions=output_attentions,
        )
        attention_output = self_attention_outputs[0]
        outputs = self_attention_outputs[1:]  # add self attentions if we output attention weights

        if self.is_decoder and encoder_hidden_states is not None:
            assert hasattr(
                self, "crossattention"
            ), f"If `encoder_hidden_states` are passed, {self} has to be instantiated with cross-attention layers by setting `config.add_cross_attention=True`"
            cross_attention_outputs = self.crossattention(
                attention_output,
                attention_mask,
                head_mask,
                encoder_hidden_states,
                encoder_attention_mask,
                output_attentions,
            )
            attention_output = cross_attention_outputs[0]
            outputs = outputs + cross_attention_outputs[1:]  # add cross attentions if we output attention weights

        layer_output = apply_chunking_to_forward(
            self.feed_forward_chunk, self.chunk_size_feed_forward, self.seq_len_dim, attention_output
        )
        outputs = (layer_output,) + outputs
        return outputs

    def feed_forward_chunk(self, attention_output):
        intermediate_output = self.intermediate(attention_output)
        layer_output = self.output(intermediate_output, attention_output)
        return layer_output


In [21]:
class CustomBertEncoder(QauntParams):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.layer = nn.ModuleList([CustomBertLayer(config) for _ in range(config.num_hidden_layers)])

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        output_attentions=False,
        output_hidden_states=False,
        return_dict=False,
    ):
        all_hidden_states = () if output_hidden_states else None
        all_self_attentions = () if output_attentions else None
        all_cross_attentions = () if output_attentions and self.config.add_cross_attention else None
        for i, layer_module in enumerate(self.layer):
            if output_hidden_states:
                all_hidden_states = all_hidden_states + (hidden_states,)

            layer_head_mask = head_mask[i] if head_mask is not None else None

            if getattr(self.config, "gradient_checkpointing", False):

                def create_custom_forward(module):
                    def custom_forward(*inputs):
                        return module(*inputs, output_attentions)

                    return custom_forward

                layer_outputs = torch.utils.checkpoint.checkpoint(
                    create_custom_forward(layer_module),
                    hidden_states,
                    attention_mask,
                    layer_head_mask,
                    encoder_hidden_states,
                    encoder_attention_mask,
                )
            else:
                layer_outputs = layer_module(
                    hidden_states,
                    attention_mask,
                    layer_head_mask,
                    encoder_hidden_states,
                    encoder_attention_mask,
                    output_attentions,
                )
            hidden_states = layer_outputs[0]
            if output_attentions:
                all_self_attentions = all_self_attentions + (layer_outputs[1],)
                if self.config.add_cross_attention:
                    all_cross_attentions = all_cross_attentions + (layer_outputs[2],)

        if output_hidden_states:
            all_hidden_states = all_hidden_states + (hidden_states,)

        if not return_dict:
            return tuple(
                v
                for v in [hidden_states, all_hidden_states, all_self_attentions, all_cross_attentions]
                if v is not None
            )
        return BaseModelOutputWithCrossAttentions(
            last_hidden_state=hidden_states,
            hidden_states=all_hidden_states,
            attentions=all_self_attentions,
            cross_attentions=all_cross_attentions,
        )


In [22]:
class CustomBertEmbeddings(QauntParams):
    """Construct the embeddings from word, position and token_type embeddings."""

    def __init__(self, config):
        super().__init__()
        self.word_embeddings = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.token_type_embeddings = nn.Embedding(config.type_vocab_size, config.hidden_size)

        # self.LayerNorm is not snake-cased to stick with TensorFlow model variable name and be able to load
        # any TensorFlow checkpoint file
        # self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.LayerNorm = QLayerNorm(config.hidden_size,
                                    in1_bits=self.nof_bits_lnorm1,
                                    in2_bits=self.nof_bits_lnorm2,
                                    eps=config.layer_norm_eps,
                                    quant=self.quant)

        self.dropout = nn.Dropout(config.hidden_dropout_prob)

        # position_ids (1, len position emb) is contiguous in memory and exported when serialized
        self.register_buffer("position_ids", torch.arange(config.max_position_embeddings).expand((1, -1)))

    def forward(self, input_ids=None, token_type_ids=None, position_ids=None, inputs_embeds=None):
        if input_ids is not None:
            input_shape = input_ids.size()
        else:
            input_shape = inputs_embeds.size()[:-1]

        seq_length = input_shape[1]

        if position_ids is None:
            position_ids = self.position_ids[:, :seq_length]

        if token_type_ids is None:
            token_type_ids = torch.zeros(input_shape, dtype=torch.long, device=self.position_ids.device)

        if inputs_embeds is None:
            inputs_embeds = self.word_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)
        token_type_embeddings = self.token_type_embeddings(token_type_ids)

        embeddings = inputs_embeds + position_embeddings + token_type_embeddings
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

In [23]:
class CustomBertModel(QBertPreTrainedModel):
    """

    The model can behave as an encoder (with only self-attention) as well as a decoder, in which case a layer of
    cross-attention is added between the self-attention layers, following the architecture described in `At\tention is
    all you need <https://arxiv.org/abs/1706.03762>`__ by Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit,
    Llion Jones, Aidan N. Gomez, Lukasz Kaiser and Illia Polosukhin.

    To behave as an decoder the model needs to be initialized with the :obj:`is_decoder` argument of the configuration
    set to :obj:`True`. To be used in a Seq2Seq model, the model needs to initialized with both :obj:`is_decoder`
    argument and :obj:`add_cross_attention` set to :obj:`True`; an :obj:`encoder_hidden_states` is then expected as an
    input to the forward pass.
    """

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config)
        self.config = config

        self.embeddings = CustomBertEmbeddings(config)
        self.encoder = CustomBertEncoder(config)

        self.pooler = CustomBertPooler(config) if add_pooling_layer else None
        self.init_weights()

    def get_input_embeddings(self):
        return self.embeddings.word_embeddings

    def set_input_embeddings(self, value):
        self.embeddings.word_embeddings = value

    def _prune_heads(self, heads_to_prune):
        """
        Prunes heads of the model. heads_to_prune: dict of {layer_num: list of heads to prune in this layer} See base
        class PreTrainedModel
        """
        for layer, heads in heads_to_prune.items():
            self.encoder.layer[layer].attention.prune_heads(heads)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        r"""
        encoder_hidden_states  (:obj:`torch.FloatTensor` of shape :obj:`(batch_size, sequence_length, hidden_size)`, `optional`):
            Sequence of hidden-states at the output of the last layer of the encoder. Used in the cross-attention if
            the model is configured as a decoder.
        encoder_attention_mask (:obj:`torch.FloatTensor` of shape :obj:`(batch_size, sequence_length)`, `optional`):
            Mask to avoid performing attention on the padding token indices of the encoder input. This mask is used in
            the cross-attention if the model is configured as a decoder. Mask values selected in ``[0, 1]``:

            - 1 for tokens that are **not masked**,
            - 0 for tokens that are **masked**.
        """
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if input_ids is not None and inputs_embeds is not None:
            raise ValueError("You cannot specify both input_ids and inputs_embeds at the same time")
        elif input_ids is not None:
            input_shape = input_ids.size()
        elif inputs_embeds is not None:
            input_shape = inputs_embeds.size()[:-1]
        else:
            raise ValueError("You have to specify either input_ids or inputs_embeds")

        device = input_ids.device if input_ids is not None else inputs_embeds.device

        if attention_mask is None:
            attention_mask = torch.ones(input_shape, device=device)
        if token_type_ids is None:
            token_type_ids = torch.zeros(input_shape, dtype=torch.long, device=device)

        # We can provide a self-attention mask of dimensions [batch_size, from_seq_length, to_seq_length]
        # ourselves in which case we just need to make it broadcastable to all heads.
        extended_attention_mask: torch.Tensor = self.get_extended_attention_mask(attention_mask, input_shape, device)

        # If a 2D or 3D attention mask is provided for the cross-attention
        # we need to make broadcastable to [batch_size, num_heads, seq_length, seq_length]
        if self.config.is_decoder and encoder_hidden_states is not None:
            encoder_batch_size, encoder_sequence_length, _ = encoder_hidden_states.size()
            encoder_hidden_shape = (encoder_batch_size, encoder_sequence_length)
            if encoder_attention_mask is None:
                encoder_attention_mask = torch.ones(encoder_hidden_shape, device=device)
            encoder_extended_attention_mask = self.invert_attention_mask(encoder_attention_mask)
        else:
            encoder_extended_attention_mask = None

        # Prepare head mask if needed
        # 1.0 in head_mask indicate we keep the head
        # attention_probs has shape bsz x n_heads x N x N
        # input head_mask has shape [num_heads] or [num_hidden_layers x num_heads]
        # and head_mask is converted to shape [num_hidden_layers x batch x num_heads x seq_length x seq_length]
        head_mask = self.get_head_mask(head_mask, self.config.num_hidden_layers)

        embedding_output = self.embeddings(
            input_ids=input_ids, position_ids=position_ids, token_type_ids=token_type_ids, inputs_embeds=inputs_embeds
        )
        encoder_outputs = self.encoder(
            embedding_output,
            attention_mask=extended_attention_mask,
            head_mask=head_mask,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_extended_attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = encoder_outputs[0]
        pooled_output = self.pooler(sequence_output) if self.pooler is not None else None

        if not return_dict:
            return (sequence_output, pooled_output) + encoder_outputs[1:]

        return BaseModelOutputWithPoolingAndCrossAttentions(
            last_hidden_state=sequence_output,
            pooler_output=pooled_output,
            hidden_states=encoder_outputs.hidden_states,
            attentions=encoder_outputs.attentions,
            cross_attentions=encoder_outputs.cross_attentions,
        )





## Bert Masked Language Modeling

In [24]:

class CustomBertPredictionHeadTransform(QauntParams):
    def __init__(self, config, quant=True):
        super().__init__()
        # self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dense = QuantizedLinear(config.hidden_size,
                    config.hidden_size,
                    nof_bits1=self.nof_bits_linear1,
                    nof_bits2=self.nof_bits_linear2,
                    quant=self.quant)

        if isinstance(config.hidden_act, str):
            self.transform_act_fn = ACT2FN[config.hidden_act]
        else:
            self.transform_act_fn = config.hidden_act
        # self.LayerNorm = nn.LayerNorm(config.hidden_size,
                                      # eps=config.layer_norm_eps)
        self.LayerNorm = QLayerNorm(config.hidden_size,
                                    in1_bits=self.nof_bits_lnorm1,
                                    in2_bits=self.nof_bits_lnorm2,
                                    eps=config.layer_norm_eps,
                                    quant=self.quant)

        # self.LayerNorm = QLayerNorm(config.hidden_size,
        #                         in1_bits=16,
        #                         in2_bits=16,
        #                         quant=quant)

    def forward(self, hidden_states):
        hidden_states = self.dense(hidden_states)
        hidden_states = self.transform_act_fn(hidden_states)
        hidden_states = self.LayerNorm(hidden_states)
        return hidden_states

class CustomBertLMPredictionHead(QauntParams):
    def __init__(self, config):
        super().__init__()
        self.transform = CustomBertPredictionHeadTransform(config)

        # The output weights are the same as the input embeddings, but there is
        # an output-only bias for each token.
        # self.decoder = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.decoder = QuantizedLinear(config.hidden_size,
                                      config.vocab_size,
                                      nof_bits1=self.nof_bits_linear1,
                                      nof_bits2=self.nof_bits_linear2,
                                      quant=self.quant)

        self.bias = nn.Parameter(torch.zeros(config.vocab_size))

        # Need a link between the two variables so that the bias is correctly resized with `resize_token_embeddings`
        self.decoder.bias = self.bias

    def forward(self, hidden_states):
        hidden_states = self.transform(hidden_states)
        hidden_states = self.decoder(hidden_states)
        return hidden_states

class CustomBertOnlyMLMHead(QauntParams):
    def __init__(self, config):
        super().__init__()
        self.predictions = CustomBertLMPredictionHead(config)

    def forward(self, sequence_output):
        prediction_scores = self.predictions(sequence_output)
        return prediction_scores


In [25]:
class CustomBertLMHeadModel(QBertPreTrainedModel):

    authorized_unexpected_keys = [r"pooler"]
    authorized_missing_keys = [r"position_ids", r"predictions.decoder.bias"]

    def __init__(self, config):
        super().__init__(config)

        # if not config.is_decoder:
        #     logger.warning("If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`")

        self.bert = CustomBertModel(config, add_pooling_layer=False)
        self.cls = CustomBertOnlyMLMHead(config)

        self.init_weights()

    def get_output_embeddings(self):
        return self.cls.predictions.decoder

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):

        # input_ids=input_ids, labels=labels
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_attention_mask,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        sequence_output = outputs[0]
        prediction_scores = self.cls(sequence_output)

        lm_loss = None
        if labels is not None:
            # we are doing next-token prediction; shift prediction scores and input ids by one
            shifted_prediction_scores = prediction_scores[:, :-1, :].contiguous()
            labels = labels[:, 1:].contiguous()
            loss_fct = CrossEntropyLoss()
            lm_loss = loss_fct(shifted_prediction_scores.view(-1, self.config.vocab_size), labels.view(-1))

        if not return_dict:
            output = (prediction_scores,) + outputs[2:]
            return ((lm_loss,) + output) if lm_loss is not None else output

        return CausalLMOutputWithCrossAttentions(
            loss=lm_loss,
            logits=prediction_scores,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
            cross_attentions=outputs.cross_attentions,
        )

    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.unset_scale_opt()


## Question Answering


In [26]:
class CustomBertForQuestionAnswering(QBertPreTrainedModel):

    authorized_unexpected_keys = [r"pooler"]

    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels

        self.bert = CustomBertModel(config, add_pooling_layer=False)
        # self.qa_outputs = nn.Linear(config.hidden_size, config.num_labels)
        self.qa_outputs = QuantizedLinear(config.hidden_size,
                                          config.num_labels,
                                          nof_bits1=self.nof_bits_linear1,
                                          nof_bits2=self.nof_bits_linear2,
                                          quant=self.quant)

        self.init_weights()


    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        start_positions=None,
        end_positions=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        r"""
        start_positions (:obj:`torch.LongTensor` of shape :obj:`(batch_size,)`, `optional`):
            Labels for position (index) of the start of the labelled span for computing the token classification loss.
            Positions are clamped to the length of the sequence (:obj:`sequence_length`). Position outside of the
            sequence are not taken into account for computing the loss.
        end_positions (:obj:`torch.LongTensor` of shape :obj:`(batch_size,)`, `optional`):
            Labels for position (index) of the end of the labelled span for computing the token classification loss.
            Positions are clamped to the length of the sequence (:obj:`sequence_length`). Position outside of the
            sequence are not taken into account for computing the loss.
        """
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        sequence_output = outputs[0]

        logits = self.qa_outputs(sequence_output)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)

        total_loss = None
        if start_positions is not None and end_positions is not None:
            # If we are on multi-GPU, split add a dimension
            if len(start_positions.size()) > 1:
                start_positions = start_positions.squeeze(-1)
            if len(end_positions.size()) > 1:
                end_positions = end_positions.squeeze(-1)
            # sometimes the start/end positions are outside our model inputs, we ignore these terms
            ignored_index = start_logits.size(1)
            start_positions.clamp_(0, ignored_index)
            end_positions.clamp_(0, ignored_index)

            loss_fct = CrossEntropyLoss(ignore_index=ignored_index)
            start_loss = loss_fct(start_logits, start_positions)
            end_loss = loss_fct(end_logits, end_positions)
            total_loss = (start_loss + end_loss) / 2

        if not return_dict:
            output = (start_logits, end_logits) + outputs[2:]
            return ((total_loss,) + output) if total_loss is not None else output

        return QuestionAnsweringModelOutput(
            loss=total_loss,
            start_logits=start_logits,
            end_logits=end_logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.unset_scale_opt()


## cola CustomBertForSequenceClassification

In [27]:
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import BertPreTrainedModel

class CustomBertForSequenceClassification(QBertPreTrainedModel):

    authorized_unexpected_keys = [r"pooler"]

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels

        self.bert = CustomBertModel(config, add_pooling_layer=True)

        # HuggingFace uses a dropout before classifier
        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        # Replace Linear with your quantized version
        self.classifier = QuantizedLinear(
            config.hidden_size,
            config.num_labels,
            nof_bits1=self.nof_bits_linear1,
            nof_bits2=self.nof_bits_linear2,
            quant=self.quant,
        )

        self.init_weights()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        # Pooled output ([CLS])
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    # ---------------- Quant control hooks ----------------
    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_scale_opt()


## CustomBertForMRPC - Sentence-pair sequence classification (MRPC)

In [28]:
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import SequenceClassifierOutput

class CustomBertForMRPC(QBertPreTrainedModel):
    """
    Sentence-pair sequence classification (MRPC).
    Inputs: (input_ids, attention_mask, token_type_ids) from tokenizer(text1, text2, ...)
    """

    authorized_unexpected_keys = [r"pooler"]

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels  # MRPC: 2

        self.bert = CustomBertModel(config, add_pooling_layer=True)

        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier = QuantizedLinear(
            config.hidden_size,
            config.num_labels,
            nof_bits1=self.nof_bits_linear1,
            nof_bits2=self.nof_bits_linear2,
            quant=self.quant,
        )

        self.init_weights()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,      # IMPORTANT for sentence pairs
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    # ---------------- Quant control hooks ----------------
    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_scale_opt()


## CustomBertForSST2

In [29]:
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import SequenceClassifierOutput

class CustomBertForSST2(QBertPreTrainedModel):
    """
    Single-sentence sentiment classification (SST-2).
    Inputs: (input_ids, attention_mask) from tokenizer(sentence)
    """

    authorized_unexpected_keys = [r"pooler"]

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels  # SST-2: 2

        self.bert = CustomBertModel(config, add_pooling_layer=True)

        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier = QuantizedLinear(
            config.hidden_size,
            config.num_labels,
            nof_bits1=self.nof_bits_linear1,
            nof_bits2=self.nof_bits_linear2,
            quant=self.quant,
        )

        self.init_weights()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    # ---------------- Quant control hooks ----------------
    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_scale_opt()


## qqCustomBertForQQP

In [30]:
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import SequenceClassifierOutput

class CustomBertForQQP(QBertPreTrainedModel):
    """
    Sentence-pair classification (QQP).
    Inputs: tokenizer(question1, question2)
    Labels:
        0 -> not duplicate
        1 -> duplicate
    """

    authorized_unexpected_keys = [r"pooler"]

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels  # QQP: 2

        self.bert = CustomBertModel(config, add_pooling_layer=True)

        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier = QuantizedLinear(
            config.hidden_size,
            config.num_labels,
            nof_bits1=self.nof_bits_linear1,
            nof_bits2=self.nof_bits_linear2,
            quant=self.quant,
        )

        self.init_weights()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,   # <-- IMPORTANT for sentence pairs
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = (
            return_dict if return_dict is not None else self.config.use_return_dict
        )

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,  # <-- sentence A/B separation
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(
                logits.view(-1, self.num_labels),
                labels.view(-1),
            )

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    # ---------------- Quant control hooks ----------------
    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_scale_opt()


## CustomBertForMNLI

In [31]:
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import SequenceClassifierOutput


class CustomBertForMNLI(QBertPreTrainedModel):
    """
    Sentence-pair classification (MNLI).

    Inputs:
        tokenizer(premise, hypothesis)

    Labels:
        0 -> contradiction
        1 -> neutral
        2 -> entailment
    """

    authorized_unexpected_keys = [r"pooler"]

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels  # MNLI: 3

        self.bert = CustomBertModel(config, add_pooling_layer=True)

        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier = QuantizedLinear(
            config.hidden_size,
            config.num_labels,
            nof_bits1=self.nof_bits_linear1,
            nof_bits2=self.nof_bits_linear2,
            quant=self.quant,
        )

        self.init_weights()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,   # REQUIRED for MNLI
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = (
            return_dict if return_dict is not None else self.config.use_return_dict
        )

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(
                logits.view(-1, self.num_labels),
                labels.view(-1),
            )

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    # ---------------- Quant control hooks ----------------
    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_scale_opt()


## CustomBertForQNLI

In [32]:
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import SequenceClassifierOutput


class CustomBertForQNLI(QBertPreTrainedModel):
    """
    Sentence-pair classification (QNLI).

    Inputs:
        tokenizer(question, sentence)

    Labels:
        0 -> not_entailment
        1 -> entailment
    """

    authorized_unexpected_keys = [r"pooler"]

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels  # QNLI: 2

        self.bert = CustomBertModel(config, add_pooling_layer=True)

        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier = QuantizedLinear(
            config.hidden_size,
            config.num_labels,
            nof_bits1=self.nof_bits_linear1,
            nof_bits2=self.nof_bits_linear2,
            quant=self.quant,
        )

        self.init_weights()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,   # REQUIRED for QNLI
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = (
            return_dict if return_dict is not None else self.config.use_return_dict
        )

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(
                logits.view(-1, self.num_labels),
                labels.view(-1),
            )

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    # ---------------- Quant control hooks ----------------
    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_scale_opt()


## CustomBertForRTE

In [33]:
import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from transformers.modeling_outputs import SequenceClassifierOutput


class CustomBertForRTE(QBertPreTrainedModel):
    """
    Sentence-pair classification (RTE).

    Inputs:
        tokenizer(premise, hypothesis)

    Labels:
        0 -> not_entailment
        1 -> entailment
    """

    authorized_unexpected_keys = [r"pooler"]

    def __init__(self, config):
        super().__init__(config)

        self.num_labels = config.num_labels  # RTE: 2

        self.bert = CustomBertModel(
            config,
            add_pooling_layer=True
        )

        classifier_dropout = (
            config.classifier_dropout
            if config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )

        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier = QuantizedLinear(
            config.hidden_size,
            config.num_labels,
            nof_bits1=self.nof_bits_linear1,
            nof_bits2=self.nof_bits_linear2,
            quant=self.quant,
        )

        self.init_weights()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,   # OPTIONAL for RTE
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = (
            return_dict if return_dict is not None else self.config.use_return_dict
        )

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,  # safe to pass None
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(
                logits.view(-1, self.num_labels),
                labels.view(-1),
            )

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    # ---------------- Quant control hooks ----------------
    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_scale_opt()


# Model Initialization

In [34]:
from transformers import (
    BertTokenizerFast,
    BertConfig,
    AutoModelForSequenceClassification

)
import torch
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    load_mem == 1
except NameError as err:

  model_name = "textattack/bert-base-uncased-RTE"

  # Tokenizer & config
  tokenizer = AutoTokenizer.from_pretrained(
    model_name
  )

  config = BertConfig.from_pretrained(model_name)

  # HuggingFace reference model
  hf_model = AutoModelForSequenceClassification.from_pretrained(model_name)
  load_mem = 1

# Your custom quantized model
model = CustomBertForRTE(config)

# Load weights
res = model.load_state_dict(hf_model.state_dict(), strict=False)
print("Missing keys:", len(res.missing_keys))
print("Unexpected keys:", len(res.unexpected_keys))
print("Missing examples:", res.missing_keys[:30])


model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Missing keys: 1
Unexpected keys: 0
Missing examples: ['bert.embeddings.position_ids']


CustomBertForRTE(
  (bert): CustomBertModel(
    (embeddings): CustomBertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): QLayerNorm(
        (768,), eps=1e-12, elementwise_affine=True
        (in_obs_normalize): MinMaxObserver()
        (in_obs): MinMaxObserver()
        (w_obs): MinMaxObserver()
        (b_obs): MinMaxObserver()
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CustomBertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CustomBertLayer(
          (attention): CustomBertAttention(
            (self): CustomBertSelfAttention(
              (query): QuantizedLinear(
                in_features=768, out_features=768, bias=True
                (in_obs): MinMaxObserver()
                (w_obs): MinMaxObserver()
                (b_obs): MinMaxObserver()
              )
              (key): QuantizedLi

## quant model

In [35]:
# q_module_list=  [QLayerNorm, IntSoftmaxTS, QuantizedLinear, QuantizedMatmul, IntGeluTS, qHadamardProd]
# q_module_list = [QLayerNorm, IntSoftmaxTS, QuantizedLinear, QuantizedMatmul]
q_module_list = [QLayerNorm]

model.set_q_module_list(q_module_list)
model.set_quant()

def audit_modes(model):
    note = []
    for name, m in model.named_modules():
        q = getattr(m, "quant", None)
        # cal = getattr(m, "is_calibrate", None)
        opt = getattr(m, "is_opt_scale", None)
        if (q is True) or (opt is True):
            note.append((name, type(m).__name__, q, opt))
    return note

note = audit_modes(model)
print("modules not in pure-float mode:", len(note))
print(*note[:50], sep="\n")


modules not in pure-float mode: 25
('bert.embeddings.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.0.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.0.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.1.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.1.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.2.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.2.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.3.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.3.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.4.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.4.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.5.attention.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer.5.output.LayerNorm', 'QLayerNorm', True, False)
('bert.encoder.layer

# Calibration

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_dataset(
    "glue",
    "rte",
    split="train",
    streaming=True
)

model.eval();

README.md: 0.00B [00:00, ?B/s]

In [37]:
# Use the streaming GLUE-RTE dataset to calibrate the quantized model
# q_module_list = [QLayerNorm, IntSoftmaxTS, QuantizedLinear, QuantizedMatmul]

if (
    QuantizedLinear in q_module_list
    or QuantizedMatmul in q_module_list
    or IntSoftmaxTS in q_module_list
    or QLayerNorm in q_module_list
):
    with torch.no_grad():
        model.set_calibration_flag()
        print()

        for i, sample in enumerate(dataset):
            if i == 100:
                break

            # GLUE-RTE sentence pairs: (sentence1, sentence2)
            s1 = sample["sentence1"]
            s2 = sample["sentence2"]

            # Defensive check
            if s1 is None or s2 is None:
                continue

            inputs = tokenizer(
                s1,
                s2,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            )

            print(i, end="|")

            input_ids = inputs["input_ids"].to(device)
            attention_mask = inputs["attention_mask"].to(device)

            token_type_ids = inputs.get("token_type_ids", None)
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            _ = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )

        model.unset_calibration_flag()
        print()
        print("Quantization parameters were set")



0|

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:1141: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


1|2|3|4|5|6|7|8|9|10|11|12|13|14|15|16|17|18|19|20|21|22|23|24|25|26|27|28|29|30|31|32|33|34|35|36|37|38|39|40|41|42|43|44|45|46|47|48|49|50|51|52|53|54|55|56|57|58|59|60|61|62|63|64|65|66|67|68|69|70|71|72|73|74|75|76|77|78|79|80|81|82|83|84|85|86|87|88|89|90|91|92|93|94|95|96|97|98|99|
Quantization parameters were set


## Optimizing input scale

In [38]:
def cosine_similarity(tensor1, tensor2):
    """Calculates cosine similarity between two tensors."""
    tensor1 = tensor1.flatten()
    tensor2 = tensor2.flatten()
    dot_product = torch.dot(tensor1, tensor2)
    norm1 = torch.norm(tensor1)
    norm2 = torch.norm(tensor2)
    return dot_product / (norm1 * norm2)

def nrmse(y_true: torch.Tensor, y_pred: torch.Tensor, eps: float = 1e-8, norm: str = "var") -> torch.Tensor:
    """
    Compute Normalized Root Mean Squared Error (NRMSE).

    Args:
        y_true: Ground truth tensor.
        y_pred: Predicted tensor (same shape as y_true).
        eps: Small constant to avoid division by zero.
        norm: Normalization method:
              - "var": normalize by sqrt(Var(y_true))  (default, scale-invariant)
              - "minmax": normalize by (max(y_true) - min(y_true))
              - "mean": normalize by mean(|y_true|)
              - "none": no normalization (just RMSE)

    Returns:
        Scalar tensor with NRMSE value.
    """
    mse = torch.mean((y_true - y_pred) ** 2)
    rmse = torch.sqrt(mse)

    if norm == "var":
        denom = torch.sqrt(y_true.var(unbiased=False) + eps)
    elif norm == "minmax":
        denom = (y_true.max() - y_true.min()).clamp(min=eps)
    elif norm == "mean":
        denom = y_true.abs().mean().clamp(min=eps)
    elif norm == "none":
        denom = 1.0
    else:
        raise ValueError(f"Unknown normalization method: {norm}")

    return rmse / denom

In [39]:
import numpy as np

def calc_cosine(layer, original_output, x, alpha=0.9, beta=1.1, num_scales=10):
    """
    Optimizes the scale factors for a layer normalization module using cosine similarity.

    Args:
        layer: The LayerNorm module whose scale factors will be optimized.
        original_output: A reference output for comparison.
        x: Input tensor to compute the cosine similarity.
        alpha: Lower bound for scale factor search.
        beta: Upper bound for scale factor search.
        num_scales: Number of scale factors to test within the [alpha, beta] range.

    Returns:
        None. Updates the layer with the optimal scale factors for weights and inputs.
    """
    # Optimize weight scale factor
    optimal_weight_scale = find_optimal_scale_factor(layer.update_w_scale_factor,
                                                     layer.forward_pass,
                                                     x,
                                                     original_output,
                                                     alpha, beta, num_scales)

    # Optimize input scale factor
    optimal_in_scale = find_optimal_scale_factor(layer.update_in_scale_factor,
                                                 layer.forward_pass,
                                                 x,
                                                 original_output,
                                                 alpha, beta, num_scales)

    # Apply the optimal scale factors to layer
    layer.set_optimal_w_scale_factor(optimal_weight_scale)
    layer.set_optimal_in_scale_factor(optimal_in_scale)


def find_optimal_scale_factor(update_scale_fn, forward_pass, x, original_output, alpha, beta, num_scales):
    """
    Finds the optimal scale factor that maximizes cosine similarity.

    Args:
        update_scale_fn: Function to update the scale factor.
        forward_pass: The forward pass function of the layer being optimized.
        x: Input tensor.
        original_output: The reference output for comparison.
        alpha: Lower bound for scale factor search.
        beta: Upper bound for scale factor search.
        num_scales: Number of scale factors to test.

    Returns:
        optimal_scale (float): The scale factor that gives the highest cosine similarity.
    """
    best_cosine = -float('inf')
    optimal_scale = 1.0

    # Test scale factors within the range [alpha, beta]
    for scale_factor in np.linspace(alpha, beta, num=num_scales):
        update_scale_fn(scale_factor)
        approx_output = forward_pass(x)
        cosine_sim = cosine_similarity(original_output, approx_output)

        if cosine_sim > best_cosine:
            best_cosine = cosine_sim
            optimal_scale = scale_factor

    return optimal_scale


In [40]:
def update_scale_factor(mdl, obs, cur_min, cur_max, search_scale):
      obs.min_val = cur_min * search_scale
      obs.max_val = cur_max * search_scale
      obs.calculate_qparams()
      mdl.is_weights_quantized = False


def optimal_factor_search(x, mdl, obs, is_weight=False, alpha=0.5, betta=2.0, itr=10, cosine_similarity=cosine_similarity):

  # sample the original scale parameters
  cur_min = obs.min_val
  cur_max = obs.max_val

  # base cosine and the optimal scale
  optimal_cosine = -1
  optimal_scale  = 1

  for search_scale in np.linspace(alpha, betta, num=itr):

      update_scale_factor(mdl, obs, cur_min, cur_max, search_scale)

      # calc cosine similarity
      approx_output = mdl.forward_pass(x)
      original_output = mdl.float_forward_pass(x)
      cosine_sim = cosine_similarity(original_output, approx_output)

      if cosine_sim > optimal_cosine:
          optimal_cosine = cosine_sim
          optimal_scale = search_scale

  # revert changes
  update_scale_factor(mdl, obs, cur_min, cur_max, 1)

  return optimal_scale

def optimal_factor_search2(x1, x2, mdl, obs, is_weight=False, alpha=0.5, betta=2.0, itr=10):

  # sample the original scale parameters
  cur_min = obs.min_val
  cur_max = obs.max_val

  # base cosine and the optimal scale
  optimal_cosine = -1
  optimal_scale  = 1

  for search_scale in np.linspace(alpha, betta, num=itr):

      update_scale_factor(mdl, obs, cur_min, cur_max, search_scale)

      # calc cosine similarity
      approx_output = mdl.forward_pass(x1, x2)
      original_output = mdl.float_forward_pass(x1, x2)
      cosine_sim = cosine_similarity(original_output, approx_output)

      if cosine_sim > optimal_cosine:
          optimal_cosine = cosine_sim
          optimal_scale = search_scale

  # revert changes
  update_scale_factor(mdl, obs, cur_min, cur_max, 1)

  return optimal_scale

def optimal_factor_search_lnorm(mdl, obs, is_weight=False, alpha=0.5, betta=2.0, itr=10):

  # sample the original scale parameters
  cur_min = obs.min_val
  cur_max = obs.max_val

  # base cosine and the optimal scale
  optimal_cosine = 100
  optimal_scale  = 1

  print("_______________________________________________")
  for search_scale in np.linspace(alpha, betta, num=itr):

      update_scale_factor(mdl, obs, cur_min, cur_max, search_scale)
      weight_integer = obs.quantizer(mdl.weight)

      # calc cosine similarity
      cosine_sim = nrmse(obs.dequantizer(weight_integer), mdl.weight)

      print(f"search_scale:{search_scale}, optimal_cosine:{optimal_cosine}, cosine_sim:{cosine_sim}")
      if (optimal_cosine - cosine_sim) > 0.1:
          optimal_cosine = cosine_sim
          optimal_scale = search_scale
  print("_______________________________________________")

  # revert changes
  update_scale_factor(mdl, obs, cur_min, cur_max, 1)
  print("-----------------")
  return optimal_scale

def insert_proxi_opt(batch, tokenizer, pair_keys):
    a = batch[pair_keys[0]]
    b = batch[pair_keys[1]]

    inputs = tokenizer(
        a,
        b,
        truncation=True,
        max_length=128,
        padding="max_length",
        return_tensors="pt",
    )

    with torch.no_grad():
        token_type_ids = inputs.get("token_type_ids", None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        _ = model(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            token_type_ids=token_type_ids,
        )

print("calibrating quantization parameters...")

PAIR_KEYS = ("sentence1", "sentence2")  # ← FIX

with torch.no_grad():
    model.set_scale_opt()
    print()

    for i, sample in enumerate(dataset):
        if i == 100:
            break

        s1 = sample["sentence1"]
        s2 = sample["sentence2"]

        if s1 is None or s2 is None:
            continue

        print(i, end="|")

        # streaming dataset → wrap as batch
        batch = {
            "sentence1": [s1],
            "sentence2": [s2],
        }

        insert_proxi_opt(batch, tokenizer, PAIR_KEYS)

        for m in model.modules():
            if type(m) not in q_module_list:
                continue

            if type(m) in [QuantizedLinear]:
                x = m.opt_input
                dq_output = m.forward_pass(x)
                output = m.float_forward_pass(x)
                cs = cosine_similarity(output, dq_output)
                if cs < (1 - 0.01):
                    print("---------------")
                    print("QuantizedLinear: cosine similarity:", f"{float(cs):.2f}")
                    search_scale_w = optimal_factor_search(x, m, m.w_obs, is_weight=True, alpha=0.5, betta=2.0, itr=10)
                    update_scale_factor(m, m.w_obs, m.w_obs.min_val, m.w_obs.max_val,    search_scale_w)
                    search_scale_in = optimal_factor_search(x, m, m.in_obs, is_weight=True, alpha=0.5, betta=2.0, itr=10)
                    update_scale_factor(m, m.in_obs, m.in_obs.min_val, m.in_obs.max_val, search_scale_in)

            # ---- LayerNorm scale optimization ----
            if type(m) is QLayerNorm and i == 0:
                x = m.opt_input

                dq_output = m.forward_pass(x)
                cs = nrmse(
                    m.w_obs.dequantizer(m.weight_integer),
                    m.weight
                )

                print(f"____________rmse:{cs}_____________")

                mean_val     = m.normalize(x)
                mean_val_f   = m.float_foward_pass_normalize(x)
                dq_output    = m.betta_gamma_forward_pass(mean_val_f)
                dq_output_f  = m.float_betta_gamma_forward_pass(mean_val_f)

                diff = dq_output - dq_output_f

                flat_idx_max = diff.argmax()
                max_idx = torch.unravel_index(flat_idx_max, diff.shape)
                flat_idx_min = diff.argmin()
                min_idx = torch.unravel_index(flat_idx_min, diff.shape)

                if cs > 0.03:
                    search_scale_w = optimal_factor_search_lnorm(
                        m,
                        m.w_obs,
                        is_weight=False,
                        alpha=0.5,
                        betta=2.0,
                        itr=10
                    )
                    print("search_scale_w", search_scale_w)
                    update_scale_factor(
                        m,
                        m.w_obs,
                        m.w_obs.min_val,
                        m.w_obs.max_val,
                        search_scale_w
                    )

    model.unset_scale_opt()
    print()



calibrating quantization parameters...

0|____________rmse:0.2348281443119049_____________
_______________________________________________
search_scale:0.5, optimal_cosine:100, cosine_sim:11.308514595031738
search_scale:0.6666666666666666, optimal_cosine:11.308514595031738, cosine_sim:4.185181617736816
search_scale:0.8333333333333333, optimal_cosine:4.185181617736816, cosine_sim:1.4013134241104126
search_scale:1.0, optimal_cosine:1.4013134241104126, cosine_sim:0.2348281443119049
search_scale:1.1666666666666665, optimal_cosine:0.2348281443119049, cosine_sim:0.25891634821891785
search_scale:1.3333333333333333, optimal_cosine:0.2348281443119049, cosine_sim:0.35618749260902405
search_scale:1.5, optimal_cosine:0.2348281443119049, cosine_sim:0.41084614396095276
search_scale:1.6666666666666665, optimal_cosine:0.2348281443119049, cosine_sim:0.41639766097068787
search_scale:1.8333333333333333, optimal_cosine:0.2348281443119049, cosine_sim:0.4289076030254364
search_scale:2.0, optimal_cosine:0.23

# Simple example - Quastion Answering Modeling

In [41]:
premise_entail = "William Shakespeare wrote the play Hamlet."
hypothesis_entail = "Shakespeare is the author of Hamlet."

premise_not_entail = "GPT is smarter than gemini"
hypothesis_not_entail = "gemini is not smarter than GPT"

import torch
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RTE_LABELS = {
    0: "entailment",
    1: "not_entailment",
}

def run_rte_example(premise, hypothesis):
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)
    pred = int(torch.argmax(logits, dim=-1).item())

    label_str = RTE_LABELS[pred]

    return {
        "premise": premise,
        "hypothesis": hypothesis,
        "prediction_id": pred,
        "label": label_str,
        "probs": probs.cpu().numpy(),
    }

# Run entailment example
res_entail = run_rte_example(premise_entail, hypothesis_entail)

# Run not-entailment example
res_not_entail = run_rte_example(premise_not_entail, hypothesis_not_entail)

print("Premise:", res_entail["premise"])
print("Hypothesis:", res_entail["hypothesis"])
print("Prediction:", res_entail["label"])
print("Probabilities [not_entailment, entailment]:", res_entail["probs"])
print()

print("Premise:", res_not_entail["premise"])
print("Hypothesis:", res_not_entail["hypothesis"])
print("Prediction:", res_not_entail["label"])
print("Probabilities [not_entailment, entailment]:", res_not_entail["probs"])


Premise: William Shakespeare wrote the play Hamlet.
Hypothesis: Shakespeare is the author of Hamlet.
Prediction: entailment
Probabilities [not_entailment, entailment]: [[0.9505153  0.04948479]]

Premise: GPT is smarter than gemini
Hypothesis: gemini is not smarter than GPT
Prediction: not_entailment
Probabilities [not_entailment, entailment]: [[0.13070586 0.8692941 ]]


In [42]:
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)


id2label: {0: 'LABEL_0', 1: 'LABEL_1'}
label2id: {'LABEL_0': 0, 'LABEL_1': 1}


# Evaluation

In [43]:
# -------------------------------------------------
# Setup
# -------------------------------------------------
import torch
from datasets import load_dataset
import evaluate
from tqdm import tqdm
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

BATCH_SIZE = 16
MAX_LEN = 128
NUM_VAL_EXAMPLES = 277  # RTE validation set size

# IMPORTANT: tokenizer from RTE checkpoint
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GLUE RTE validation set (streaming)
dataset = load_dataset(
    "glue",
    "rte",
    split="validation",
    streaming=True,
).shuffle(seed=400)

streamer = dataset.iter(batch_size=BATCH_SIZE)

# RTE metric: accuracy
metric = evaluate.load("glue", "rte")

predictions = []
references = []

# =================================================
# Batch inference
# =================================================
def get_batch_predictions(batch, tokenizer):
    s1 = batch["sentence1"]
    s2 = batch["sentence2"]

    inputs = tokenizer(
        s1,
        s2,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    token_type_ids = inputs.get("token_type_ids", None)
    if token_type_ids is not None:
        token_type_ids = token_type_ids.to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )

    logits = outputs.logits          # [B, 2]
    preds = torch.argmax(logits, dim=-1).cpu().tolist()
    return preds


# -------------------------------------------------
# Evaluation loop
# -------------------------------------------------
for i, batch in enumerate(tqdm(streamer, total=NUM_VAL_EXAMPLES // BATCH_SIZE)):

    if i * BATCH_SIZE >= NUM_VAL_EXAMPLES:
        break

    # Defensive filtering
    valid_idx = [
        j for j, (a, b) in enumerate(zip(batch["sentence1"], batch["sentence2"]))
        if a is not None and b is not None
    ]
    if len(valid_idx) == 0:
        continue

    filtered_batch = {
        "sentence1": [batch["sentence1"][j] for j in valid_idx],
        "sentence2": [batch["sentence2"][j] for j in valid_idx],
        "label":     [batch["label"][j]     for j in valid_idx],
    }

    batch_preds = get_batch_predictions(filtered_batch, tokenizer)

    for j in range(len(batch_preds)):
        predictions.append(batch_preds[j])
        references.append(filtered_batch["label"][j])

    # ✅ Interim accuracy every 10 batches
    if (i + 1) % 10 == 0:
        res = metric.compute(
            predictions=predictions,
            references=references,
        )
        print(
            f"\n[Batch {i+1}] "
            f"Accuracy: {res['accuracy']:.4f}\n"
        )

# -------------------------------------------------
# Final metric
# -------------------------------------------------
res = metric.compute(
    predictions=predictions,
    references=references,
)

print(
    "Final Results | "
    f"Accuracy: {res['accuracy']:.4f}"
)


 65%|██████▍   | 11/17 [00:01<00:00,  8.56it/s]


[Batch 10] Accuracy: 0.7250



18it [00:02,  7.22it/s]                        

Final Results | Accuracy: 0.7148


In [44]:
# # -------------------------------------------------
# # Setup
# # -------------------------------------------------
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# hf_model.eval()
# hf_model.to(device)

# BATCH_SIZE = 16
# MAX_LEN = 128
# NUM_VAL_EXAMPLES = 1043  # full CoLA validation set

# tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# # GLUE CoLA validation set (streaming)
# dataset = load_dataset(
#     "glue",
#     "cola",
#     split="validation",
#     streaming=True
# ).shuffle(seed=400)

# streamer = dataset.iter(batch_size=BATCH_SIZE)

# # CoLA metric = Matthews Correlation Coefficient
# metric = evaluate.load("glue", "cola")

# predictions = []
# references = []

# # =================================================

# def get_batch_predictions(batch, tokenizer):
#     inputs = tokenizer(
#         batch["sentence"],
#         truncation=True,
#         padding="max_length",
#         max_length=MAX_LEN,
#         return_tensors="pt",
#     )

#     input_ids = inputs["input_ids"].to(device)
#     attention_mask = inputs["attention_mask"].to(device)

#     with torch.no_grad():
#         try:
#             outputs = hf_model(
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#             )

#         except RuntimeError as err:
#           print(err)
#           raise RuntimeError(err)

#     logits = outputs.logits
#     preds = torch.argmax(logits, dim=-1).cpu().tolist()
#     return preds


# # Iterate through batches
# for i, batch in enumerate(tqdm(streamer, total=NUM_VAL_EXAMPLES // BATCH_SIZE)):

#     if i * BATCH_SIZE >= NUM_VAL_EXAMPLES:
#         break

#     batch_preds = get_batch_predictions(batch, tokenizer)

#     for j in range(len(batch_preds)):
#         predictions.append(batch_preds[j])
#         references.append(batch["label"][j])

#     # ✅ Interim MCC every 10 batches
#     if (i + 1) % 10 == 0:
#         partial_results = metric.compute(
#             predictions=predictions,
#             references=references,
#         )
#         print(
#             f"\n[Batch {i+1}] Interim MCC: "
#             f"{partial_results['matthews_correlation']:.4f}\n"
#         )

# results = metric.compute(
#     predictions=predictions,
#     references=references,
# )

# print("Final MCC:", results["matthews_correlation"])


# LNORM STATS

In [45]:
# Models
#     # CoLA - geckos/bert-base-uncased-finetuned-glue-cola
#     # MRPC - lrs21/bert-base-uncased-finetuned-glue-mrpc
#     # SST-2 - raj-p/bert-base-uncased-finetuned-glue-sst2
#     # QQP - textattack/bert-base-uncased-QQP
#     # MNLI - textattack/bert-base-uncased-MNLI
#     # QNLI - mrm8488/bert-uncased-finetuned-qnli
#     # RTE - nickapch/bert-base-uncased-finetuned-glue_rte
#     # SQuAD - bert-large-uncased-whole-word-masking-finetuned-squad

# results
# baseline     - Accuracy: 0.7256
# all          - Accuracy: 0.7076
# all but linear - Accuracy: 0.7148
# norm no opt  - Accuracy: 0.5523
# only norm    - Accuracy: 0.7148
